# Language model and text generation

In this lab, we will teach a recurrent network to write "in the style of Voltaire", character by character: from the 30 previous characters, the model predicts the probability of each next character. It is a language model, here at the character level.

1. A character-level language model.
2. Generating text, with a temperature.
3. The training, looking at what the model writes.
4. The effect of the temperature.
5. LSTM, GRU or simple RNN?
6. Two LSTM layers.

This lab needs a GPU: check in `Runtime > Change runtime type` that the hardware accelerator is a GPU. Its trainings take about 4 minutes in all there on an RTX 3090; Colab's T4 is about two to three times slower.

## Getting the data

The corpus' texts by Voltaire come from [gutenberg.org](https://www.gutenberg.org), without their headers, licenses and prefaces: our language model mustn't learn to write Gutenberg's legal notices or the publishers' prefaces. We only use `voltaire/voltaire_clean.txt`, which gathers them. The cell shows its first 1,000 characters.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh voltaire
!ls -l voltaire/
!head -c 1000 voltaire/voltaire_clean.txt

## Importing the libraries

In [ ]:
import pathlib

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

# Same random draws at each run: reproducible results
keras.utils.set_random_seed(0)

## Preparing the data

The next cell prepares the examples:

- the text, in lowercase, in `text`;
- the vocabulary: the text's characters, sorted, in `index_to_char` (index → character), and `char_to_index` (character → index); `encode` turns a string into an array of indices;
- the examples: sequences of 30 characters (`maxlen`), taken every 3 characters; the target of each one is the character that follows it;
- `X`, the encoded sequences, of shape `(number of sequences, 30)`, and `y`, the indices of the next characters.

In [ ]:
text = pathlib.Path("voltaire/voltaire_clean.txt").read_text(encoding="utf8").lower()
index_to_char = sorted(set(text))
char_to_index = {char: index for index, char in enumerate(index_to_char)}
print(f"{len(text)} characters: {text[:80]}…")
print(f"{len(index_to_char)} distinct characters: {''.join(index_to_char)}")


def encode(characters: str) -> np.ndarray:
  return np.array([char_to_index[char] for char in characters])


maxlen = 30
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i : i + maxlen])
  next_chars.append(text[i + maxlen])

X = np.stack([encode(sentence) for sentence in sentences])
y = encode(next_chars)
print(f"Shapes of X and y: {X.shape}, {y.shape}")
print(f'First example: "{sentences[0]}" → "{next_chars[0]}", {X[0]} → {y[0]}')

## 1. A character-level language model

Write the function `build_model(*recurrent_layers)`, which returns a compiled `keras.Sequential` model:

- a `keras.Input` input: sequences of characters of any length, made of integer indices;
- a `layers.Embedding` layer: a vector of 8 numbers for each character of the vocabulary;
- the recurrent layers `recurrent_layers`, in order;
- a `layers.Dense` layer that gives a probability to each character of the vocabulary;
- compiled with the loss `sparse_categorical_crossentropy` (the targets are indices), the Adam optimizer with clipping of the gradient's norm to 1, and the accuracy as metric.

The next parts call it with the recurrent layer(s) to compare, first `build_model(layers.LSTM(128))`.

In [ ]:
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  # Your code here
  ...

### Additional hints

The calls and the parameters to use:

- `keras.Input(shape=(None,), dtype="int32")`
- `layers.Embedding(len(index_to_char), 8)`: no `mask_zero` here, the sequences have no padding
- `*recurrent_layers` in a list inserts the layers one by one
- `layers.Dense(len(index_to_char), activation="softmax")`
- `keras.optimizers.Adam(clipnorm=1.0)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=..., dtype=...),
      layers.Embedding(..., ...),
      *recurrent_layers,
      layers.Dense(..., activation=...),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=...),
    loss=...,
    metrics=["accuracy"],
  )
  return model
```

### Solution

In [ ]:
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=(None,), dtype="int32"),  # sequences of any length
      layers.Embedding(len(index_to_char), 8),
      *recurrent_layers,
      layers.Dense(len(index_to_char), activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=1.0),  # gradient clipping
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model

The next cell builds the lab's model: an LSTM with 128 units.

In [ ]:
model = build_model(layers.LSTM(128))
model.summary()

## 2. Generating text, with a temperature

Write the function `generate(model, seed, n=200, temperature=1.0)`, which returns the text `seed` followed by `n` characters generated by `model`, one by one:

1. encode the last `maxlen` characters of the text, and give them to the model as a batch of a single sequence;
2. the model returns the probabilities $p_i$ of the next character; apply the temperature $T$ to them, as in the slides: $p_i \propto \exp(z_i / T)$, where the logits $z_i$ are $\log p_i$ up to a constant, that is $p_i^{1/T}$, normalized so that they sum to 1;
3. draw the next character according to these probabilities, and add it to the text.

The cell after the solution tries it on the model not trained yet.

**Useful documentation**

- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)

In [ ]:
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  # Your code here
  ...

### Additional hints

The calls to use:

- `generated[-maxlen:]`: the last `maxlen` characters; `encode(...)[None, :]` adds the batch dimension, of size 1
- `model(x, training=False)`: a direct call, faster than `model.predict` for a single sequence; it returns a tensor of shape `(1, len(index_to_char))`, which `keras.ops.convert_to_numpy` turns into a NumPy array
- convert the probabilities to `float64` before raising them to the power `1 / temperature`: in `float32`, their sum differs enough from 1 for `np.random.choice` to refuse the draw
- `np.random.choice(len(probas), p=probas)` draws an index according to the probabilities `probas`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  generated = seed
  for _ in range(n):
    x = encode(generated[...])[None, :]
    probas = keras.ops.convert_to_numpy(model(x, training=False))[0]
    probas = probas.astype("float64") ** ...
    probas /= ...
    generated += index_to_char[np.random.choice(len(probas), p=...)]
  return generated
```

### Solution

In [ ]:
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  generated = seed
  for _ in range(n):
    # The last maxlen characters, as a batch of one sequence
    x = encode(generated[-maxlen:])[None, :]
    probas = keras.ops.convert_to_numpy(model(x, training=False))[0]
    # p_i ∝ exp(z_i / T), with z_i = log p_i up to a constant
    probas = probas.astype("float64") ** (1 / temperature)
    probas /= probas.sum()
    generated += index_to_char[np.random.choice(len(probas), p=probas)]
  return generated

Before training, the model gives roughly the same probability to all the characters: it writes gibberish.

In [ ]:
print(generate(model, text[:maxlen], n=100))

## 3. Training the model

The callback `SamplingCallback` makes the model write 200 characters every `every` epochs, from a randomly drawn excerpt of the text: we thus see the model improve.

The end of the corpus, 10% of the sequences (`validation_split=0.1`), is for validation: Keras takes it at the end of the data, without shuffling. A random split would put almost the same sequences on both sides, since two neighboring sequences share 27 characters.

The training lasts 20 epochs: about a minute on an RTX 3090, two to three on Colab's T4.

In [ ]:
class SamplingCallback(keras.callbacks.Callback):
  def __init__(self, every: int = 5, n: int = 200) -> None:
    super().__init__()
    self.every = every
    self.n = n

  def on_epoch_end(self, epoch: int, logs: dict | None = None) -> None:
    if (epoch + 1) % self.every == 0:
      start = np.random.randint(len(text) - maxlen)
      seed = text[start : start + maxlen]
      print(f'\nAfter {epoch + 1} epochs, from "{seed}":')
      # self.model: the model fit is training
      print(generate(self.model, seed, n=self.n))


history = model.fit(
  X,
  y,
  batch_size=512,
  epochs=20,
  validation_split=0.1,
  callbacks=[SamplingCallback(every=5)],
)

## 4. The effect of the temperature

With the trained model, generate 300 characters from the same start of sentence, for instance `"candide était "`, with the temperatures 0.2, 0.5, 1 and 1.5. How does the text change when the temperature goes down? when it goes up?

In [ ]:
# Your code here

### Additional hints

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Temperature {temperature}:")
  print(generate(model, "candide était ", n=..., temperature=...))
  print()
```

### Solution

In [ ]:
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Temperature {temperature}:")
  print(generate(model, "candide était ", n=300, temperature=temperature))
  print()

At a low temperature (0.2), the model almost always picks the most probable character: it writes short and frequent words, real or nearly so ("monde", "contre", but also "parde"), and soon loops on "de la … de la …". At a high temperature (1.5), the distribution flattens: unlikely characters come out more often, and the text is nothing but a string of syllables, made-up words and signs ("&", "?", ";"). In between, around 0.5, the text is the most plausible: more varied words and fewer repetitions.

None of these texts means anything: after 20 epochs, the model only predicts the next character right about one time in two (47% validation accuracy). It has learned the short words, French endings ("-ait", "-ent") and punctuation, not yet the long words nor the meaning.

## 5. LSTM, GRU or simple RNN?

Train three models with 128 units, an LSTM, a GRU and a simple RNN (`layers.SimpleRNN`), for 10 epochs each, and compare their validation losses (`val_loss`), for instance on the same plot. Which cell does best? Does the simple RNN lag behind, as the vanishing gradient would suggest?

In [ ]:
# Your code here

### Additional hints

- `build_model(layers.GRU(128))`: the same model, with a GRU
- `fit` returns a `History` object, whose dictionary `history` gives the validation loss of each epoch, under the key `"val_loss"`
- the same `fit` parameters as in part 3, without the callback

A skeleton to copy into the exercise cell, replacing each `...`:

```python
histories = {}
for name, layer in [("LSTM", layers.LSTM(128)), ("GRU", ...), ("SimpleRNN", ...)]:
  histories[name] = build_model(...).fit(
    X, y, batch_size=512, epochs=10, validation_split=0.1
  )

for name, cell_history in histories.items():
  plt.plot(cell_history.history[...], label=name)
plt.xlabel("Epoch")
plt.ylabel(...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
histories = {}
for name, layer in [
  ("LSTM", layers.LSTM(128)),
  ("GRU", layers.GRU(128)),
  ("SimpleRNN", layers.SimpleRNN(128)),
]:
  histories[name] = build_model(layer).fit(
    X, y, batch_size=512, epochs=10, validation_split=0.1
  )

for name, cell_history in histories.items():
  plt.plot(cell_history.history["val_loss"], label=name)
  print(f"{name}: minimum validation loss {min(cell_history.history['val_loss']):.3f}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.legend()
plt.show()

After 10 epochs, the GRU is clearly ahead (minimum validation loss about 1.63, better than the LSTM of part 3 in 20 epochs), and the simple RNN (1.78) even beats the LSTM (1.87). None of the three curves has finished going down: this ranking mostly tells which cell learns fastest. The LSTM has the most weights to tune (70,144, against 52,992 for the GRU and 17,536 for the simple RNN), and it starts slowly.

The simple RNN doesn't lag behind here: 30 characters make a short sequence, and the vanishing gradient (it fades from one time step to the next) doesn't hinder it much yet. It penalizes it on longer sequences, where the gates of the LSTM and the GRU let the information, and the gradient, through over longer distances. Gradient clipping (`clipnorm`) protects it from the other risk, the exploding gradient.

These models haven't overfit yet: their validation loss still decreases. With many more epochs, or a bigger model, it would end up rising while the training one still decreases: that would be the time to stop the training (callback [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/)).

## 6. Two LSTM layers

A last model, given: two stacked LSTMs. The first layer returns one state per character (`return_sequences=True`), the sequence the second one reads. Compare its validation loss with the one of the model of part 3, and what it writes.

In [ ]:
deep_model = build_model(layers.LSTM(128, return_sequences=True), layers.LSTM(128))
deep_model.summary()
deep_history = deep_model.fit(
  X,
  y,
  batch_size=512,
  epochs=20,
  validation_split=0.1,
  callbacks=[SamplingCallback(every=10)],
)
print(
  f"Minimum validation loss: one LSTM {min(history.history['val_loss']):.3f}, "
  f"two {min(deep_history.history['val_loss']):.3f}"
)

The two layers do better than the single LSTM of part 3: after 20 epochs, a minimum validation loss of about 1.66 against 1.74. They cost 2.6 times more parameters (211,455 against 79,871) and longer epochs. The text, though, is still made of right short words and made-up long ones: to write sentences, a character-level language model needs much more training.